In [1]:
!pip install transformers torch datasets scikit-learn shap lime pandas numpy matplotlib tqdm -q

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 275.7/275.7 kB 9.7 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


In [2]:
import os
if os.path.exists('phishing_email.csv'):
    print('Dataset found!')
else:
    print('Dataset not found. Please upload phishing_email.csv')

Dataset found!


In [3]:
import pandas as pd

# Path to the dataset, as used by step2_preprocess.py
data_file_path = 'phishing_email.csv'

# Load the dataset, rename the problematic column, and save it back
print(f"Attempting to preprocess {data_file_path} for column name compatibility.")
df_fix_cols = pd.read_csv(data_file_path)
if 'text_combined' in df_fix_cols.columns and 'text' not in df_fix_cols.columns:
    df_fix_cols.rename(columns={'text_combined': 'text'}, inplace=True)
    df_fix_cols.to_csv(data_file_path, index=False)
    print("Column 'text_combined' renamed to 'text' in the dataset.")
else:
    print("Dataset already contains 'text' column or 'text_combined' not found. Skipping rename.")

# Now run the original preprocessing script
%run step2_preprocess.py

Attempting to preprocess phishing_email.csv for column name compatibility.
Column 'text_combined' renamed to 'text' in the dataset.
Loading dataset from phishing_email.csv...
Columns found: ['text', 'label']
Shape: (20258, 2)

--- Starting preprocessing ---
Initial shape: (20258, 2)
Label distribution:
label
0.0    12986
1.0     7271
Name: count, dtype: int64

Cleaning text...


/content/step2_preprocess.py:170: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['label'] = df['label'].astype(int)
/content/step2_preprocess.py:174: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['text'] = df['text'].apply(clean_text)


After removing short texts: (20231, 2)
After removing duplicates: (19902, 2)

Balancing classes to 7233 samples each...
Balanced dataset shape: (14466, 2)
Label distribution after balancing:
label
1    7233
0    7233
Name: count, dtype: int64

Split sizes:
  Train: 10126 (70.0%)
  Val:   1445 (10.0%)
  Test:  2895 (20.0%)

Saved to processed_data/
Files: train.csv, val.csv, test.csv

Preprocessing complete. Run step3_train_transformers.py next.


In [4]:
!sed -i 's/evaluation_strategy/eval_strategy/g' step3_train_transformers.py
!sed -i 's/save_strategy/save_strategy/g' step3_train_transformers.py

In [5]:
!sed -i 's/base_estimator/estimator/g' step4_baselines.py

In [6]:
!sed -i 's|ROBERTA_PATH      = "/content/checkpoints/roberta_phishing"|ROBERTA_PATH      = "./checkpoints/roberta_phishing"|g' step5_xai.py
!sed -i 's|BERT_PATH         = "/content/checkpoints/bert_phishing"|BERT_PATH         = "./checkpoints/bert_phishing"|g' step5_xai.py

In [7]:
import os
for root, dirs, files in os.walk('./checkpoints'):
    for f in files:
        print(os.path.join(root, f))

In [8]:
import os
for root, dirs, files in os.walk('./checkpoints'):
    print(root)
    for f in files:
        print('  ', f)

In [9]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

train_df = pd.read_csv('processed_data/train.csv')
val_df   = pd.read_csv('processed_data/val.csv')
test_df  = pd.read_csv('processed_data/test.csv')

class EmailDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=512, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze(), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds  = np.argmax(logits, axis=-1)
    probs  = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:,1]
    acc    = accuracy_score(labels, preds)
    p,r,f,_ = precision_recall_fscore_support(labels, preds, average='binary')
    auc    = roc_auc_score(labels, probs)
    return {'accuracy':round(acc,4),'precision':round(p,4),'recall':round(r,4),'f1':round(f,4),'auc_roc':round(auc,4)}

print("Loading RoBERTa...")
tokenizer = AutoTokenizer.from_pretrained('roberta-base')
model     = AutoModelForSequenceClassification.from_pretrained('roberta-base', num_labels=2)

args = TrainingArguments(
    output_dir='checkpoints/roberta_phishing',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=RANDOM_SEED,
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=EmailDataset(train_df, tokenizer),
    eval_dataset=EmailDataset(val_df, tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Training RoBERTa — this takes 20-40 minutes...")
trainer.train()
trainer.save_model('checkpoints/roberta_phishing')
tokenizer.save_pretrained('checkpoints/roberta_phishing')
print("RoBERTa saved!")

# Print test results
results = trainer.evaluate(EmailDataset(test_df, tokenizer))
print("\n--- RoBERTa Test Results ---")
for k,v in results.items():
    if 'runtime' not in k:
        print(f"  {k}: {v}")

Loading RoBERTa...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/481 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['roberta.embeddings.LayerNorm.weight', 'roberta.embeddings.LayerNorm.bias', 'roberta.encoder.layer.0.attention.output.LayerNorm.weight', 'roberta.encoder.layer.0.attention.output.LayerNorm.bias', 'roberta.encoder.layer.0.output.LayerNorm.weight', 'roberta.encoder.layer.0.output.LayerNorm.bias', 'roberta.encoder.layer.1.attention.output.LayerNorm.weight', 'roberta.encoder.layer.1.attention.output.LayerNorm.bias', 'roberta.encoder.layer.1.output.LayerNorm.weight', 'roberta.encoder.layer.1.output.LayerNorm.bias', 'roberta.encoder.layer.2.attention.output.LayerNorm.weight', 'roberta.encoder.layer.2.attention.output.LayerNorm.bias', 'roberta.encoder.layer.2.output.LayerNorm.weight', 'roberta.encoder.layer.2.output.LayerNorm.bias', 'roberta.encoder.layer.3.attention.output.LayerNorm.weight', 'roberta.encoder.layer.3.attention.output.LayerNorm.bias', 'roberta.encoder.layer.3.output.LayerNorm.weight', 'roberta.encoder.layer.3.output.Laye

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

RoBERTa saved!



--- RoBERTa Test Results ---
  eval_loss: 0.07772062718868256
  eval_accuracy: 0.9889
  eval_precision: 0.991
  eval_recall: 0.9869
  eval_f1: 0.9889
  eval_auc_roc: 0.9995
  eval_samples_per_second: 118.324
  eval_steps_per_second: 7.398
  epoch: 4.0


In [12]:
import pandas as pd
import torch
import numpy as np
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer, EarlyStoppingCallback
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, roc_auc_score

RANDOM_SEED = 42
torch.manual_seed(RANDOM_SEED)

train_df = pd.read_csv('processed_data/train.csv')
val_df   = pd.read_csv('processed_data/val.csv')
test_df  = pd.read_csv('processed_data/test.csv')

class EmailDataset(Dataset):
    def __init__(self, df, tokenizer):
        self.texts  = df['text'].tolist()
        self.labels = df['label'].tolist()
        self.tok    = tokenizer
    def __len__(self): return len(self.texts)
    def __getitem__(self, idx):
        enc = self.tok(self.texts[idx], max_length=512, padding='max_length', truncation=True, return_tensors='pt')
        return {'input_ids': enc['input_ids'].squeeze(), 'attention_mask': enc['attention_mask'].squeeze(), 'labels': torch.tensor(self.labels[idx], dtype=torch.long)}

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:,1]
    acc = accuracy_score(labels, preds)
    p,r,f,_ = precision_recall_fscore_support(labels, preds, average='binary')
    auc = roc_auc_score(labels, probs)
    return {'accuracy':round(acc,4),'precision':round(p,4),'recall':round(r,4),'f1':round(f,4),'auc_roc':round(auc,4)}

print("Loading BERT...")
tokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')
model = AutoModelForSequenceClassification.from_pretrained('bert-base-uncased', num_labels=2)

args = TrainingArguments(
    output_dir='checkpoints/bert_phishing',
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    warmup_ratio=0.1,
    eval_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='f1',
    seed=RANDOM_SEED,
    fp16=True,
    report_to='none',
)

trainer = Trainer(
    model=model, args=args,
    train_dataset=EmailDataset(train_df, tokenizer),
    eval_dataset=EmailDataset(val_df, tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

print("Training BERT — 20-40 minutes...")
trainer.train()
trainer.save_model('checkpoints/bert_phishing')
tokenizer.save_pretrained('checkpoints/bert_phishing')
print("BERT saved!")

results = trainer.evaluate(EmailDataset(test_df, tokenizer))
print("\n--- BERT Test Results ---")
for k,v in results.items():
    if 'runtime' not in k:
        print(f"  {k}: {v}")

Loading BERT...


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

BERT saved!



--- BERT Test Results ---
  eval_loss: 0.052634917199611664
  eval_accuracy: 0.99
  eval_precision: 0.9883
  eval_recall: 0.9917
  eval_f1: 0.99
  eval_auc_roc: 0.9994
  eval_samples_per_second: 108.187
  eval_steps_per_second: 6.764
  epoch: 4.0


In [14]:
%run step4_baselines.py

Loading preprocessed data...
Train: 10126, Test: 2895

Training SVM baseline

Running grid search for SVM...
  Parameter grid: {'clf__estimator__C': [0.1, 1.0, 10.0]}
Fitting 3 folds for each of 3 candidates, totalling 9 fits
  Best params: {'clf__estimator__C': 1.0}
  Best CV F1:  0.9910

--- SVM (TF-IDF) Test Results ---
  Accuracy:  0.9938 (99.38%)
  Precision: 0.9911
  Recall:    0.9965
  F1-score:  0.9938
  AUC-ROC:   0.9995

  Confusion Matrix:
    TP=1442  FP=13
    FN=5  TN=1435

  Full Classification Report:
              precision    recall  f1-score   support

  Legitimate       1.00      0.99      0.99      1448
    Phishing       0.99      1.00      0.99      1447

    accuracy                           0.99      2895
   macro avg       0.99      0.99      0.99      2895
weighted avg       0.99      0.99      0.99      2895

SVM saved.

Training Random Forest baseline

Running grid search for Random Forest...
  Parameter grid: {'clf__n_estimators': [100, 200], 'clf__max_de

In [13]:
%run step5_xai.py

Using device: cuda
Loading test data...
Test phishing emails available: 1447
Loading model from checkpoints/roberta_phishing...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading model from checkpoints/bert_phishing...


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Running SHAP on 1447 phishing emails...


...

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 100%|██████████| 50/50 [06:25<00:00, 11.39s/it]

  0%|          | 0/498 [00:00<?, ?it/s]

PartitionExplainer explainer: 51it [06:37,  7.96s/it]


Saving Figure 2 (SHAP token map)...
  Saved: figures//figure2_shap_tokens.png

Top 10 SHAP tokens (global):
  mailing              0.3260
  sometimes            0.3155
  post                 0.3143
  education            0.3105
  better               0.3028
  experiences          0.3007
  http                 0.2330
  com                  0.1885
  info                 0.1650
  buttermagma          0.1317

Running LIME on 1447 phishing emails...
  LIME progress: 0/100
  LIME progress: 10/100
  LIME progress: 20/100
  LIME progress: 30/100
  LIME progress: 40/100
  LIME progress: 50/100
  LIME progress: 60/100
  LIME progress: 70/100
  LIME progress: 80/100
  LIME progress: 90/100

Top 10 LIME tokens (aggregated):
  better               0.2028
  privet               0.1373
  webcam               0.1255
  sometimes            0.1150
  free                 0.1103
  raised               0.0836
  born                 0.0822
  buttermagma          0.0779
  manipulate           0.0746
  google